In [ ]:
# @title 1. Install Dependencies
!pip install -q transformers accelerate torch sentence-transformers scikit-learn tqdm pandas numpy

In [ ]:
# @title 2. Imports and Configuration
import re
import json
import torch
import numpy as np
import pandas as pd
from datetime import datetime
from typing import List, Dict, Tuple, Optional
from collections import defaultdict, Counter
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Configuration
MODEL_NAME = "athena129/CyberSecQwen-4B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LOGS_FOR_ANALYSIS = 10000
MAX_GROUPS_TO_ANALYZE = 5
EMBEDDING_BATCH_SIZE = 32

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
    def analyze_group(self, group: Dict) -> str:
        """Analyze log group using LLM with structured data (OPTIMIZED)"""

        # Prepare structured summary
        event_type = group.get('event_type', 'UNKNOWN')
        size = group['size']
        levels = group.get('levels', {})

        # Extract common fields from group (limit for speed)
        field_summary = defaultdict(Counter)
        for log in group.get('logs', group.get('samples', []))[:10]:  # Limit to 10 samples
            for field_name, field_value in log.get('fields', {}).items():
                field_summary[field_name][field_value] += 1

        # Get most common values for each field
        common_fields = {}
        for field_name, counter in field_summary.items():
            common_fields[field_name] = counter.most_common(2)

        # Prepare sample messages (limit to 2)
        samples_text = "\n".join([f"- {log['message'][:200]}"
                                  for log in group.get('samples', [])[:2]])

        # Prepare field summary text
        fields_text = ""
        for field_name, values in common_fields.items():
            values_str = ", ".join([f"{val} ({count})" for val, count in values])
            fields_text += f"  {field_name}: {values_str}\n"

        prompt = f"""Analyze security events:
Type: {event_type}, Count: {size}, Levels: {levels}

Fields:
{fields_text if fields_text else "  None"}

Samples:
{samples_text}

Answer briefly:
1. Attack type
2. Severity (low/medium/high/critical)
3. Actions needed"""

        try:
            # Format prompt for Qwen chat
            messages = [
                {"role": "system", "content": "You are a cybersecurity expert. Be concise."},
                {"role": "user", "content": prompt}
            ]

            formatted_prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = self.tokenizer(formatted_prompt, return_tensors="pt", truncation=True, max_length=256)
            if DEVICE == "cuda":
                inputs = {k: v.cuda() for k, v in inputs.items()}

            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=150,  # Reduced from 400 to 150
                    temperature=0.3,     # Reduced for faster generation
                    do_sample=False,     # Greedy decoding is faster
                    num_beams=1,         # No beam search
                    pad_token_id=self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id
                )

            # Decode only the new tokens
            input_length = inputs['input_ids'].shape[1]
            response_tokens = outputs[0][input_length:]
            response = self.tokenizer.decode(response_tokens, skip_special_tokens=True)

            # Clean up
            response = response.strip()

            if len(response) < 10:
                return self._generate_fallback_analysis(group)

            return response

        except Exception as e:
            print(f"Error: {e}")
            return self._generate_fallback_analysis(group)

In [ ]:
# @title 4. Analysis Function
def analyze_log_content(analyzer: LogAnalyzer, content: str) -> Dict:
    """Complete log analysis pipeline"""

    print("\n" + "="*60)
    print("LOG ANALYSIS")
    print("="*60)

    # Parse logs
    important_logs = analyzer.parse_logs(content)

    # Statistics
    level_stats = defaultdict(int)
    for log in important_logs:
        level_stats[log['level']] += 1

    print(f"\nTotal important events: {len(important_logs)}")
    print("Severity distribution:")

    severity_emoji = {'CRITICAL': '⛔', 'ERROR': '🔴', 'WARNING': '⚠️'}
    for level, count in sorted(level_stats.items(), key=lambda x: -x[1]):
        print(f"  {severity_emoji.get(level, '•')} {level}: {count}")

    if not important_logs:
        print("\nNo important events found!")
        return {}

    # Security event type statistics
    event_type_stats = defaultdict(int)
    for log in important_logs:
        event_type = log.get('event_type', 'UNKNOWN')
        event_type_stats[event_type] += 1

    print("\nSecurity Event Types:")
    total_important = len(important_logs)
    for event_type, count in sorted(event_type_stats.items(), key=lambda x: -x[1]):
        percentage = (count / total_important) * 100 if total_important > 0 else 0
        print(f"  🎯 {event_type}: {count} ({percentage:.1f}%)")

    # N-gram analysis
    print("\nAnalyzing common patterns (n-grams)...")
    ngram_analysis = analyzer.analyze_ngrams(important_logs, top_k=20)

    print("\nTop patterns:")
    for ngram in ngram_analysis['top_ngrams']:
        print(f"  • {ngram['pattern']}: {ngram['count']}")

    # Group similar logs
    print("\nGrouping logs...")
    groups = analyzer.group_similar_logs(important_logs)
    print(f"Found {len(groups)} groups")

    # Analyze top groups
    results = []
    max_groups = min(MAX_GROUPS_TO_ANALYZE, len(groups))

    print(f"\nAnalyzing top {max_groups} groups...")

    for i, group in enumerate(groups[:max_groups]):
        event_type = group.get('event_type', 'UNKNOWN')
        print(f"\n  Analyzing group {i+1}/{max_groups} ({event_type}, {group['size']} events)...")
        analysis = analyzer.analyze_group(group)

        results.append({
            'group_id': group['group_id'],
            'event_type': event_type,
            'group_size': group['size'],
            'levels': group['levels'],
            'samples': [log['message'][:200] for log in group['samples'][:2]],
            'analysis': analysis
        })

    return {
        'total_lines': len(content.strip().split('\n')),
        'total_important': len(important_logs),
        'level_stats': dict(level_stats),
        'event_type_stats': dict(event_type_stats),
        'ngram_analysis': ngram_analysis,
        'total_groups': len(groups),
        'analyzed_groups': results
    }

In [ ]:
# @title 5. Upload and Analyze Log File
from google.colab import files
import json

# Initialize analyzer
analyzer = LogAnalyzer()

print("\n" + "="*60)
print("LOG FILE UPLOAD")
print("="*60)
print("\nUpload your .log file:")
print("Supported formats: Generic, Apache, Syslog, or any text with keywords")
print("Keywords: error, critical, warning, fail, exception")

uploaded = files.upload()

results = None

for filename in uploaded.keys():
    print(f"\n{'='*60}")
    print(f"File: {filename}")
    print(f"{'='*60}")

    # Read file
    try:
        with open(filename, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()

        line_count = len(content.strip().split('\n'))
        print(f"File size: {line_count} lines")

        # Analyze
        results = analyze_log_content(analyzer, content)

        # Save results
        if results:
            output_file = f"analysis_{filename.replace('.log', '').replace('.txt', '')}.json"
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=2, default=str)
            print(f"\nResults saved: {output_file}")

    except Exception as e:
        print(f"Error processing file: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
# @title 6. Display Analysis Results
import matplotlib.pyplot as plt

def display_results(results: Dict):
    """Display analysis results in a readable format"""

    if not results:
        print("No results to display")
        return

    print("\n" + "="*70)
    print("ANALYSIS REPORT")
    print("="*70)

    print(f"\nTotal lines: {results.get('total_lines', 'N/A')}")
    print(f"Important events: {results.get('total_important', 0)}")
    print(f"Error groups: {results.get('total_groups', 0)}")

    # Severity statistics
    print("\n" + "-"*40)
    print("SEVERITY LEVELS")
    print("-"*40)

    severity_emoji = {'CRITICAL': '⛔', 'ERROR': '🔴', 'WARNING': '⚠️'}
    level_stats = results.get('level_stats', {})

    # Filter out None keys
    level_stats = {k: v for k, v in level_stats.items() if k is not None}

    for level, count in sorted(level_stats.items(), key=lambda x: -x[1]):
        percentage = (count / results['total_important']) * 100 if results['total_important'] > 0 else 0
        print(f"  {severity_emoji.get(level, '•')} {level}: {count} ({percentage:.1f}%)")

    # Security event types
    event_type_stats = results.get('event_type_stats', {})

    # Filter out None keys
    event_type_stats = {k: v for k, v in event_type_stats.items() if k is not None}

    if event_type_stats:
        print("\n" + "-"*40)
        print("SECURITY EVENT TYPES")
        print("-"*40)

        for event_type, count in sorted(event_type_stats.items(), key=lambda x: -x[1]):
            percentage = (count / results['total_important']) * 100 if results['total_important'] > 0 else 0
            print(f"  🎯 {event_type}: {count} ({percentage:.1f}%)")

    # Visualization - Severity
    if level_stats:
        try:
            plt.figure(figsize=(8, 5))
            levels = [str(l) for l in level_stats.keys()]  # Convert to string
            counts = [int(c) for c in level_stats.values()]  # Convert to int

            colors = ['#ff4444' if 'CRITICAL' in l else
                     '#ff6b6b' if 'ERROR' in l else
                     '#ffa500' for l in levels]

            plt.bar(levels, counts, color=colors)
            plt.title('Log Severity Distribution', fontsize=14, fontweight='bold')
            plt.ylabel('Count')
            plt.xticks(rotation=45)
            plt.grid(axis='y', alpha=0.3)
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Visualization error (severity): {e}")

    # Visualization - Event Types
    if event_type_stats:
        try:
            plt.figure(figsize=(10, 6))
            event_types = [str(t) for t in event_type_stats.keys()]  # Convert to string
            event_counts = [int(c) for c in event_type_stats.values()]  # Convert to int

            plt.bar(event_types, event_counts, color='#4CAF50')
            plt.title('Security Event Types', fontsize=14, fontweight='bold')
            plt.ylabel('Count')
            plt.xticks(rotation=45, ha='right')
            plt.grid(axis='y', alpha=0.3)
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Visualization error (event types): {e}")

    # N-gram analysis
    ngram_analysis = results.get('ngram_analysis', {})
    if ngram_analysis and ngram_analysis.get('top_ngrams'):
        print("\n" + "-"*40)
        print("COMMON PATTERNS (N-GRAMS)")
        print("-"*40)

        for ngram in ngram_analysis['top_ngrams'][:10]:
            pattern = ngram.get('pattern', 'Unknown')
            count = ngram.get('count', 0)
            print(f"  • {pattern}: {count}")

    # Group analysis
    print("\n" + "="*70)
    print("DETAILED GROUP ANALYSIS")
    print("="*70)

    analyzed_groups = results.get('analyzed_groups', [])

    for i, group in enumerate(analyzed_groups):
        print(f"\n{'─'*70}")
        event_type = group.get('event_type', 'UNKNOWN')
        if event_type is None:
            event_type = 'UNKNOWN'

        print(f"Group {i+1}: {event_type}")
        print(f"Size: {group.get('group_size', 0)} events")
        print(f"Levels: {group.get('levels', {})}")

        print("\nExamples:")
        samples = group.get('samples', [])
        for sample in samples[:2]:  # Limit to 2 samples for readability
            if sample:
                print(f"  • {sample[:150]}")  # Truncate long samples

        print("\nExpert Analysis:")
        analysis_text = group.get('analysis', 'No analysis available')
        if analysis_text:
            # Truncate very long analysis
            if len(analysis_text) > 500:
                print(f"  {analysis_text[:500]}...")
            else:
                print(f"  {analysis_text}")
        else:
            print("  No analysis available")

        print(f"{'─'*70}")

# Display results if available
if 'results' in locals() and results:
    display_results(results)
else:
    print("Run the analysis first (previous cell)")